In [3]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score
import category_encoders as ce

# 1. Load raw data
train = pd.read_csv('Data/train.csv')
test = pd.read_csv('Data/test.csv')

# 2. Create the Adversarial Target
train['Is_Test'] = 0
test['Is_Test'] = 1
adv_data = pd.concat([train, test], ignore_index=True)

# Drop the real target and IDs
adv_y = adv_data['Is_Test']
adv_X = adv_data.drop(columns=['Id', 'Drafted', 'Is_Test'])

# 3. Quick Preprocessing (Encode categoricals so LGBM can read them)
cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']
for col in cat_cols:
    adv_X[col] = adv_X[col].astype(str)

encoder = ce.OrdinalEncoder(cols=cat_cols)
adv_X_encoded = encoder.fit_transform(adv_X)

# 4. Train Adversarial Model
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
adv_model = LGBMClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Get Out-Of-Fold predictions
adv_preds = cross_val_predict(adv_model, adv_X_encoded, adv_y, cv=cv, method='predict_proba')[:, 1]
adv_auc = roc_auc_score(adv_y, adv_preds)

print(f"--- ADVERSARIAL VALIDATION AUC: {adv_auc:.5f} ---")

if adv_auc > 0.60:
    print("\n[!] WARNING: Severe Train/Test Drift Detected.")
    # Fit once on all data to get feature importances
    adv_model.fit(adv_X_encoded, adv_y)
    importances = pd.DataFrame({
        'Feature': adv_X.columns,
        'Importance': adv_model.feature_importances_
    }).sort_values(by='Importance', ascending=False)

    print("\nTop 10 Drifting Features (Consider dropping or neutralizing these):")
    print(importances.head(10))
else:
    print("\n[+] SAFE: Train and Test sets have similar distributions. Drift is not your main problem.")

[LightGBM] [Info] Number of positive: 557, number of negative: 2224
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000277 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 981
[LightGBM] [Info] Number of data points in the train set: 2781, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.200288 -> initscore=-1.384497
[LightGBM] [Info] Start training from score -1.384497
[LightGBM] [Info] Number of positive: 556, number of negative: 2225
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000248 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 971
[LightGBM] [Info] Number of data points in the train set: 2781, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.199928 -> initscore=-1.386744
[LightGBM] [Info] Start training from score -1.386744
[LightGBM] [Info] Numb

In [7]:
def engineer_global_features(df, train_df_for_counts=None):
    df = df.copy()
    df['Weight_lbs'] = df['Weight'] * 2.20462
    df['Height_inches'] = df['Height'] * 39.3701

    combine_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                    'Broad_Jump', 'Agility_3cone', 'Shuttle']

    # Missingness Profile
    df['Combine_Missing_Count'] = df[combine_cols].isnull().sum(axis=1)
    df['Did_Any_Agility'] = ((df['Agility_3cone'].notnull()) | (df['Shuttle'].notnull())).astype(int)

    # Safe Math for Physics
    safe_40yd = df['Sprint_40yd'].replace(0, np.nan).fillna(999)
    safe_height = df['Height_inches'].replace(0, np.nan).fillna(999)
    safe_weight = df['Weight_lbs'].replace(0, np.nan).fillna(1)

    # Advanced Physics / Scouting Composites
    df['BMI'] = (df['Weight_lbs'] / (safe_height ** 2)) * 703
    df['Speed_Score'] = np.where(df['Sprint_40yd'] > 0, (df['Weight_lbs'] * 200) / (safe_40yd ** 4), 0)
    df['Burst_Score'] = np.where((df['Vertical_Jump'] > 0) & (df['Broad_Jump'] > 0), df['Vertical_Jump'] + df['Broad_Jump'], 0)
    df['Power_Success'] = np.where(df['Bench_Press_Reps'] > 0, df['Bench_Press_Reps'] * df['Weight_lbs'], 0)

    # Momentum (Mass x Velocity Proxy)
    df['Momentum'] = df['Weight_lbs'] / safe_40yd
    # Explode proxy (How far can you jump per lb of bodyweight)
    df['Broad_per_lb'] = df['Broad_Jump'].fillna(0) / safe_weight

    cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']
    for col in cat_cols:
        df[col] = df[col].fillna('Unknown').astype(str)

    return df

# --- IMPORTANT EXECUTIONS OUTSIDE THE FUNCTION ---
train_df = engineer_global_features(train)
test_df = engineer_global_features(test)

# THE SCHOOL TIER HACK (Fit strictly on Train, mapped to both)
school_counts = train_df['School'].value_counts()
train_df['School_Volume'] = train_df['School'].map(school_counts)
test_df['School_Volume'] = test_df['School'].map(school_counts).fillna(1) # Unseen schools get 1

# Interaction: Missingness x School Prestige (The "Skipped Drill Privilege")
train_df['Missing_x_Volume'] = train_df['Combine_Missing_Count'] * np.log1p(train_df['School_Volume'])
test_df['Missing_x_Volume'] = test_df['Combine_Missing_Count'] * np.log1p(test_df['School_Volume'])

y = train_df['Drafted']
X = train_df.drop(columns=['Id', 'Drafted', 'Weight', 'Height'])
X_test_base = test_df.drop(columns=['Id', 'Weight', 'Height'])